In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import os

directory = "/users/cdcook/VSP/graphics/"

sns.set_theme(style="darkgrid")

# Function to check if a specific flag is true
def flag_is_true(row, flag):
    return row.get(flag, False)

def oneDFit(a,b,x):
    return a*x+b

field = 'sky0001_1a'
night = '0906'
subimageDir = '/lustre/work/client/users/cdcook/VSPData/subimages'

In [ ]:
# Define the objInfoFlag dictionary with swapped keys and values
objInfoFlags = {
    0x00000000: "DEFAULT",
    0x00000001: "FEW",
    0x00000002: "POOR",
    0x00000004: "ICRF_QSO",
    0x00000008: "HERN_QSO_P60",
    0x00000010: "HERN_QSO_P05",
    0x00000020: "HERN_RRL_P60",
    0x00000040: "HERN_RRL_P05",
    0x00000080: "HERN_VARIABLE",
    0x00000100: "TRANSIENT",
    0x00000200: "HAS_SOLSYS_DET",
    0x00000400: "MOST_SOLSYS_DET",
    0x00000800: "LARGE_PM",
    0x00001000: "RAW_AVE",
    0x00002000: "FIT_AVE",
    0x00004000: "FIT_PM",
    0x00008000: "FIT_PAR",
    0x00010000: "USE_AVE",
    0x00020000: "USE_PM",
    0x00040000: "USE_PAR",
    0x00080000: "NO_MEAN_ASTROM",
    0x00100000: "STACK_FOR_MEAN",
    0x00200000: "MEAN_FOR_STACK",
    0x00400000: "BAD_PM",
    0x00800000: "EXT",
    0x01000000: "EXT_ALT",
    0x02000000: "GOOD",
    0x04000000: "GOOD_ALT",
    0x08000000: "GOOD_STACK",
    0x10000000: "BEST_STACK",
    0x20000000: "SUSPECT_STACK",
    0x40000000: "BAD_STACK"
}

def read_flags(flag_value, flag_dict):
    """Function to read and interpret flag values based on the provided flag dictionary."""
    active_flags = {flag_dict[flag_hex]: (flag_value & flag_hex) != 0 for flag_hex in flag_dict}
    return {flag_name: active for flag_name, active in active_flags.items() if active}

def add_flag_descriptions(df, flag_col_name, flag_dict):
    """Add a description column to the DataFrame based on the flags."""
    descriptions = df[flag_col_name].apply(lambda x: read_flags(int(x, 16) if isinstance(x, str) else x, flag_dict))
    description_col_name = f"{flag_col_name}_Description"
    df[description_col_name] = descriptions
    return df

In [ ]:
def KronCut(kronBand, kronDist, dataDF):
    if (kronBand == 'g'):
        kronName = 'gMeanKronMag'
        psfName = 'gMeanPSFMag'
        kronCutName = 'gKron'
    if (kronBand == 'r'):
        kronName = 'rMeanKronMag'
        psfName = 'rMeanPSFMag'
        kronCutName = 'rKron'
    if (kronBand == 'i'):
        kronName = 'iMeanKronMag'
        psfName = 'iMeanPSFMag'
        kronCutName = 'iKron'
    if (kronBand == 'z'):
        kronName = 'zMeanKronMag'
        psfName = 'zMeanPSFMag'
        kronCutName = 'zKron'
    if (kronBand == 'y'):
        kronName = 'yMeanKronMag'
        psfName = 'yMeanPSFMag'
        kronCutName = 'yKron'
        
    # Condition 1: psfName - kronName should be less than kronDist
    condition1 = abs(dataDF[psfName] - dataDF[kronName]) < kronDist
    # Apply both conditions to the DataFrame
    dataDF = dataDF[condition1]
    
    return dataDF, kronName, psfName, kronCutName

def BitFlagCut(bitFlags, dataDF):
    dataDF = dataDF[dataDF['Flags'] == bitFlags]
    return dataDF

def ColorCut(dataDF, sub1, sub2):
    if (sub1 == 'gr'):
        band1a = 'gMeanPSFMag'
        band1b = 'rMeanPSFMag'
        band1name = "g-r"
        
    elif (sub1 == 'gi'):
        band1a = 'gMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "g-i"
        
    elif (sub1 == 'ri'):
        band1a = 'rMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "r-i"
        
    if (sub2 == 'gr'):
        band2a = 'gMeanPSFMag'
        band2b = 'rMeanPSFMag'
        band2name = "g-r"
    
    elif (sub2 == 'gi'):
        band2a = 'gMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "g-i"
        
    elif (sub2 == 'ri'):
        band2a = 'rMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "r-i"
        
    x = []
    y = []
    for index, row in dataDF.iterrows():
        x.append(row[band1a] - row[band1b])
        y.append(row[band2a] - row[band2b])

    dataDF.insert(0, 'color1', x)
    dataDF.insert(1, 'color2', y)
    return dataDF, band1a, band1b, band2a, band2b, band1name, band2name

In [ ]:
exposure = 54
title = f'/lustre/work/client/users/cdcook/VSPData/meanFields/mean{field}{night}Files/{field}00{night}_exp' + str(exposure) + '.csv' #sky0001_1c000928_exp0.csv
#title = f'/lustre/work/client/users/cdcook/VSPData/meanFields/mean{field}{night}Files/Pan00{night}{field}_exp' + str(exposure) + '.csv' #Pan000409xtetrans_1b_exp0.csv
panDF = pd.read_csv(title)
panDF = panDF.drop_duplicates(subset=['RA', 'Dec'])
panDF = panDF[(panDF['Mag'] >= 5) & (panDF['Mag'] <= 25)]
panDF = panDF[panDF['gMeanPSFMag'] != -999]
panDF = panDF[panDF['rMeanPSFMag'] != -999]
panDF = panDF[panDF['iMeanPSFMag'] != -999]
panDF = panDF[panDF['zMeanPSFMag'] != -999]
panDF = panDF[panDF['yMeanPSFMag'] != -999]
panDF = panDF[panDF['gMeanKronMag'] != -999]
panDF = panDF[panDF['rMeanKronMag'] != -999]
panDF = panDF[panDF['iMeanKronMag'] != -999]
panDF = panDF[panDF['zMeanKronMag'] != -999]
panDF = panDF[panDF['yMeanKronMag'] != -999]

# Define a function to calculate the flux
def calculate_flux(magnitude):
    return np.power(10, (magnitude + 48.6) / -2.5)

# Calculate flux for each band
panDF['gflux'] = panDF['gMeanPSFMag'].apply(calculate_flux)
panDF['rflux'] = panDF['rMeanPSFMag'].apply(calculate_flux)
panDF['iflux'] = panDF['iMeanPSFMag'].apply(calculate_flux)
panDF['zflux'] = panDF['zMeanPSFMag'].apply(calculate_flux)
panDF['yflux'] = panDF['yMeanPSFMag'].apply(calculate_flux)

# Only do this if you need to
panDF['RA'] = panDF['RA'].apply(lambda ra: ra + 360 if ra < 180 else ra)

# Calculate total flux
panDF['totalFlux'] = (
    panDF['gflux'] * 0.1212 + 
    panDF['rflux'] * 0.1463 + 
    panDF['iflux'] * 0.1435 + 
    panDF['zflux'] * 0.098 + 
    panDF['yflux'] * 0.0393
) / 0.5483

# Calculate logpart
panDF['logpart'] = np.log10(panDF['totalFlux'] / 3631e-23)

# Calculate pseudoBoloMag
panDF['pseudoBoloMag'] = -2.5 * panDF['logpart']

flux_columns = ['gflux', 'rflux', 'iflux', 'zflux', 'yflux']
panDF.drop(columns=flux_columns, inplace=True)

panDF['Difference'] = panDF['pseudoBoloMag'] - panDF['Mag']

panDF['objInfoFlag'] = panDF['objInfoFlag'].apply(lambda x: f'0x{x:08X}')
panDF['qualityFlag'] = panDF['qualityFlag'].apply(lambda x: f'0x{x:08X}')

print(panDF)
panDF = add_flag_descriptions(panDF, "objInfoFlag", objInfoFlags)

In [ ]:
bitFlags = 0.0
color2 = 'gr'
color1 = 'ri'
kronBand = 'r'
kronDist = '0.5'

kronCutTF = False
bitFlagTF = False
colorCutTF = False


panDF, band1a, band1b, band2a, band2b, band1name, band2name = ColorCut(panDF, 'gr', 'ri')
panDF, kronName, psfName, kronCutName = KronCut(kronBand='g', kronDist = 0.5, dataDF = panDF)
kronCutTF = True
print(len(panDF))

In [ ]:
def subimage_by_ra_dec_balanced(df, n_ra_div=10, n_dec_div=10):
    """Split the dataframe into subimages by evenly dividing the RA and Dec ranges."""
    ra_min, ra_max = df['RA'].min(), df['RA'].max()
    dec_min, dec_max = df['Dec'].min(), df['Dec'].max()
    
    ra_bins = np.linspace(ra_min, ra_max, n_ra_div + 1)
    dec_bins = np.linspace(dec_min, dec_max, n_dec_div + 1)
    
    subimages = []
    
    for i in range(n_ra_div):
        for j in range(n_dec_div):
            ra_mask = (df['RA'] >= ra_bins[i]) & (df['RA'] < ra_bins[i + 1])
            dec_mask = (df['Dec'] >= dec_bins[j]) & (df['Dec'] < dec_bins[j + 1])
            sub_df = df[ra_mask & dec_mask]
            subimages.append(sub_df)
    
    return subimages

# Split the data into subimages
subimages = subimage_by_ra_dec_balanced(panDF, n_ra_div=10, n_dec_div=10)

# Table to store RA/Dec limits, slope, AB offset, and count for each subimage
summary_table = []

for idx, sub_df in enumerate(subimages):
    ra_min, ra_max = sub_df['RA'].min(), sub_df['RA'].max()
    dec_min, dec_max = sub_df['Dec'].min(), sub_df['Dec'].max()
    
    if not sub_df.empty:
        # Fit a line to the data
        try:
            params = np.polyfit(sub_df['Mag'], sub_df['pseudoBoloMag'], 1)
            slope, ab_offset = params[0], params[1]
        except Exception as e:
            print(f"Error fitting line for subimage {idx + 1}: {e}")
            slope, ab_offset = np.nan, np.nan
    else:
        slope, ab_offset = np.nan, np.nan
    
    # Save subimage details to summary table
    summary_table.append({
        'Subimage': idx + 1,
        'RA_min': ra_min,
        'RA_max': ra_max,
        'Dec_min': dec_min,
        'Dec_max': dec_max,
        'Slope': slope,
        'AB_offset': ab_offset,
        'Count': sub_df.shape[0]
    })

# Create a summary DataFrame from the table
summary_df = pd.DataFrame(summary_table)
print("\nSummary Table:")
print(summary_df)

subimageDir = '/lustre/work/client/users/cdcook/VSPData/subimages'

csvName = f"{subimageDir}/{field}{night}subimage_exp_{str(exposure)}.csv"

# Save summary table to a CSV file
summary_df.to_csv(csvName, index=False)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Load the summary DataFrame (adjust path if necessary)
csvName = f"{subimageDir}/{field}{night}subimage_exp_{str(exposure)}.csv"
summary_df = pd.read_csv(csvName)

# Create a 3D plot
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')

# Choose what to plot on the z-axis (e.g., slope, offset, or count)
z_axis = 'Count'  # Change to 'Slope' or 'AB_offset' as needed

# Plot the data
sc = ax.scatter(summary_df['RA_min'] + (summary_df['RA_max'] - summary_df['RA_min']) / 2,
                summary_df['Dec_min'] + (summary_df['Dec_max'] - summary_df['Dec_min']) / 2,
                summary_df[z_axis],
                c=summary_df[z_axis],
                cmap='viridis',
                s=50)

# Set labels
ax.set_xlabel('RA')
ax.set_ylabel('Dec')
ax.set_zlabel(z_axis)
ax.set_title(f'3D Plot of {z_axis}')

# Add a color bar
cbar = plt.colorbar(sc)
cbar.set_label(z_axis)

plt.show()

In [ ]:
# Reshape the data for heatmap
def reshape_for_heatmap(df, n_ra_div=10, n_dec_div=10):
    ra_bins = np.linspace(df['RA_min'].min(), df['RA_max'].max(), n_ra_div + 1)
    dec_bins = np.linspace(df['Dec_min'].min(), df['Dec_max'].max(), n_dec_div + 1)
    
    # Create empty matrices for each parameter
    slope_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    offset_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    count_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    
    for _, row in df.iterrows():
        ra_idx = np.digitize(row['RA_min'], ra_bins) - 1
        dec_idx = np.digitize(row['Dec_min'], dec_bins) - 1
        if 0 <= ra_idx < n_ra_div and 0 <= dec_idx < n_dec_div:
            slope_matrix[ra_idx, dec_idx] = row['Slope']
            offset_matrix[ra_idx, dec_idx] = row['AB_offset']
            count_matrix[ra_idx, dec_idx] = row['Count']
    
    return ra_bins, dec_bins, slope_matrix, offset_matrix, count_matrix

# Assuming you have the `summary_df` DataFrame from earlier
ra_bins, dec_bins, slope_matrix, offset_matrix, count_matrix = reshape_for_heatmap(summary_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

# Define a function to format tick labels
def format_ticks(ticks):
    return [f'{tick:.3g}' for tick in ticks]

# Plot heatmaps
sns.heatmap(slope_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[0], cbar_kws={'label': 'Slope'}, vmin=0.0, vmax=1.0)
axes[0].set_title(f'Slope Heatmap {field} {night} -- exposure {exposure}')
axes[0].invert_yaxis()

sns.heatmap(offset_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[1], cbar_kws={'label': 'AB Offset'}, vmin=0.0, vmax=10.0)
axes[1].set_title(f'AB Offset Heatmap {field} {night} -- exposure {exposure}')
axes[1].invert_yaxis()

sns.heatmap(count_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[2], cbar_kws={'label': 'Count'}, vmin=0.0, vmax=160)
axes[2].set_title(f'Count Heatmap {field} {night} -- exposure {exposure}')
axes[2].invert_yaxis()

plt.show()

In [ ]:
# Reshape the data for heatmap and calculate difference from the mean
def reshape_for_heatmap(df, n_ra_div=10, n_dec_div=10):
    ra_bins = np.linspace(df['RA_min'].min(), df['RA_max'].max(), n_ra_div + 1)
    dec_bins = np.linspace(df['Dec_min'].min(), df['Dec_max'].max(), n_dec_div + 1)
    
    # Calculate overall mean and standard deviation
    overall_slope_mean = df['Slope'].mean()
    overall_offset_mean = df['AB_offset'].mean()
    overall_count_mean = df['Count'].mean()

    overall_slope_std = df['Slope'].std()
    overall_offset_std = df['AB_offset'].std()
    overall_count_std = df['Count'].std()

    # Create empty matrices for each parameter and their differences from the mean
    slope_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    offset_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    count_matrix = np.full((n_ra_div, n_dec_div), np.nan)

    slope_diff_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    offset_diff_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    count_diff_matrix = np.full((n_ra_div, n_dec_div), np.nan)
    
    for ra_idx in range(n_ra_div):
        for dec_idx in range(n_dec_div):
            ra_mask = (df['RA_min'] >= ra_bins[ra_idx]) & (df['RA_min'] < ra_bins[ra_idx+1])
            dec_mask = (df['Dec_min'] >= dec_bins[dec_idx]) & (df['Dec_min'] < dec_bins[dec_idx+1])
            bin_data = df[ra_mask & dec_mask]
            
            if not bin_data.empty:
                slope_mean = bin_data['Slope'].mean()
                offset_mean = bin_data['AB_offset'].mean()
                count_mean = bin_data['Count'].mean()

                slope_matrix[ra_idx, dec_idx] = slope_mean
                offset_matrix[ra_idx, dec_idx] = offset_mean
                count_matrix[ra_idx, dec_idx] = count_mean

                slope_diff_matrix[ra_idx, dec_idx] = slope_mean - overall_slope_mean
                offset_diff_matrix[ra_idx, dec_idx] = offset_mean - overall_offset_mean
                count_diff_matrix[ra_idx, dec_idx] = count_mean - overall_count_mean

    return (ra_bins, dec_bins, slope_matrix, offset_matrix, count_matrix, 
            slope_diff_matrix, offset_diff_matrix, count_diff_matrix,
            overall_slope_std, overall_offset_std, overall_count_std)

# Assuming you have the `summary_df` DataFrame from earlier
(ra_bins, dec_bins, slope_matrix, offset_matrix, count_matrix,
 slope_diff_matrix, offset_diff_matrix, count_diff_matrix,
 overall_slope_std, overall_offset_std, overall_count_std) = reshape_for_heatmap(summary_df)

# Print the overall standard deviations
print("Slope Standard Deviation:", overall_slope_std)
print("Offset Standard Deviation:", overall_offset_std)
print("Count Standard Deviation:", overall_count_std)

fig, axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)

# Define a function to format tick labels
def format_ticks(ticks):
    return [f'{tick:.3g}' for tick in ticks]

# Plot actual value heatmaps
sns.heatmap(slope_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[0, 0], cbar_kws={'label': 'Slope'}, vmin=0.0, vmax=1.0)
axes[0, 0].set_title(f'Slope Heatmap {field} {night} -- exposure {exposure}')
axes[0, 0].invert_yaxis()

sns.heatmap(offset_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[0, 1], cbar_kws={'label': 'AB Offset'}, vmin=0.0, vmax=10.0)
axes[0, 1].set_title(f'AB Offset Heatmap {field} {night} -- exposure {exposure}')
axes[0, 1].invert_yaxis()

sns.heatmap(count_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="viridis", ax=axes[0, 2], cbar_kws={'label': 'Count'}, vmin=0.0, vmax=160)
axes[0, 2].set_title(f'Count Heatmap {field} {night} -- exposure {exposure}')
axes[0, 2].invert_yaxis()

# Plot difference from the mean heatmaps
sns.heatmap(slope_diff_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="coolwarm", ax=axes[1, 0], cbar_kws={'label': 'Slope Difference from Mean'})
axes[1, 0].set_title(f'Slope Difference from Mean {field} {night} -- exposure {exposure}')
axes[1, 0].invert_yaxis()

sns.heatmap(offset_diff_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="coolwarm", ax=axes[1, 1], cbar_kws={'label': 'AB Offset Difference from Mean'})
axes[1, 1].set_title(f'AB Offset Difference from Mean {field} {night} -- exposure {exposure}')
axes[1, 1].invert_yaxis()

sns.heatmap(count_diff_matrix, xticklabels=format_ticks(ra_bins), yticklabels=format_ticks(dec_bins), cmap="coolwarm", ax=axes[1, 2], cbar_kws={'label': 'Count Difference from Mean'})
axes[1, 2].set_title(f'Count Difference from Mean {field} {night} -- exposure {exposure}')
axes[1, 2].invert_yaxis()

plt.show()